# Supervised Trajectories — Reject and Revise, and Abort

This notebook shows two trajectories available with `SupervisedTaskHandler`
that are not possible with `run()` or `run(with_approval=True)`:

- **Reject and Revise.** After the agent delivers its result, the human calls
  `reject()` with structured feedback. The handler routes the rejection back
  through `get_next_step()` deterministically — the agent revises without
  replanning from scratch.
- **Abort.** The human calls `abort()` mid-execution to terminate cleanly,
  setting a `TaskHandlerError` (or a custom exception) on the handler that
  the caller can inspect.

Both trajectories use a code refactoring scenario to keep the focus on the
HITL mechanics rather than tooling.

In [ ]:
# Uncomment the line below to install `llm-agents-from-scratch` from PyPI
# !pip install llm-agents-from-scratch

## Running an Ollama service

To execute the code provided in this notebook, you'll need to have Ollama
installed on your local machine and have its LLM hosting service running.
To download Ollama, follow the instructions found on this page:
https://ollama.com/download. After downloading and installing Ollama, you
can start a service by opening a terminal and running `ollama serve`.

In [1]:
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request


def ensure_ollama(host="http://localhost:11434", timeout=15):
    """Start Ollama if not already running and wait until responsive."""

    def _up():
        try:
            urllib.request.urlopen(f"{host}/api/tags", timeout=1)
            return True
        except (urllib.error.URLError, ConnectionError, TimeoutError):
            return False

    if _up():
        return print(f"\u2713 Ollama already running at {host}")

    ollama_path = shutil.which("ollama")
    if ollama_path is None:
        for candidate in [
            "/teamspace/studios/this_studio/.local/bin/ollama",
            "/usr/local/bin/ollama",
            "/usr/bin/ollama",
        ]:
            if os.path.exists(candidate):
                ollama_path = candidate
                break
    if ollama_path is None:
        raise RuntimeError(
            "Could not find the ollama binary. Install with: "
            "curl -fsSL https://ollama.com/install.sh | sh",
        )

    print(f"Starting Ollama server ({ollama_path})...")
    subprocess.Popen(
        [ollama_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _up():
            return print(f"\u2713 Ollama up and running at {host}")
        time.sleep(0.5)

    raise RuntimeError(f"Ollama did not start within {timeout}s")


ensure_ollama()

✓ Ollama already running at http://localhost:11434


In [2]:
import logging

from llm_agents_from_scratch import LLMAgent
from llm_agents_from_scratch.data_structures import Task
from llm_agents_from_scratch.llms import OllamaLLM
from llm_agents_from_scratch.logger import enable_console_logging

enable_console_logging(logging.INFO)

llm = OllamaLLM(model="qwen3:14b", think=False)
agent = LLMAgent(llm=llm)

## Trajectory 1: Reject and Revise

The agent is asked to refactor a function by replacing its `if-elif` chain
with a dictionary dispatch. After it delivers its result, the human inspects
it, finds it incomplete (no error handling), and calls `reject()` with
specific feedback. The handler feeds the rejection back through
`get_next_step()` and the agent revises.

In [3]:
ORIGINAL = """\
def calculate(x, y, operation):
    if operation == \"add\":
        return x + y
    elif operation == \"subtract\":
        return x - y
    elif operation == \"multiply\":
        return x * y
    elif operation == \"divide\":
        return x / y
"""

In [4]:
task = Task(
    instruction=(
        "Refactor the following Python function to replace the "
        f"if-elif chain with a dictionary dispatch:\n\n{ORIGINAL}"
    ),
)
task_handler = await agent.run_supervised(task)

In [5]:
step = await task_handler.get_next_step(None)
step

TaskStep(id_='6861d83d-ba0d-4ff4-bfff-e4362b184f19', task_id='830884ad-95a4-4fd7-9002-67deb56e4fba', instruction='Refactor the following Python function to replace the if-elif chain with a dictionary dispatch:\n\ndef calculate(x, y, operation):\n    if operation == "add":\n        return x + y\n    elif operation == "subtract":\n        return x - y\n    elif operation == "multiply":\n        return x * y\n    elif operation == "divide":\n        return x / y\n')

In [6]:
step_result = await task_handler.run_step(step)
step_result

INFO (llm_agents_fs.SupervisedTaskHandler) :      ⚙️ Processing Step: Refactor the following Python function to replace the if-elif chain with a dictionary dispatch:

def calculate(x, y, operation):
 ...[TRUNCATED]
INFO (llm_agents_fs.SupervisedTaskHandler) :      ✅ Step Result: I need to refactor the given function by replacing the if-elif chain with a dictionary dispatch. This means I will create a dictionary ...[TRUNCATED]


TaskStepResult(task_step_id='6861d83d-ba0d-4ff4-bfff-e4362b184f19', content="I need to refactor the given function by replacing the if-elif chain with a dictionary dispatch. This means I will create a dictionary that maps operation strings to corresponding lambda functions or function references. Then, I will use the dictionary to look up the appropriate function based on the operation parameter. Let's implement this approach.")

In [7]:
step = await task_handler.get_next_step(step_result)
step

In [8]:
step_result = await task_handler.run_step(step)
step_result

In [9]:
result = await task_handler.get_next_step(step_result)
result

In [10]:
rejected = task_handler.reject(
    result,
    feedback=(
        "Good start, but add error handling: raise ValueError for "
        "unknown operations and ZeroDivisionError for division by zero."
    ),
)
rejected

RejectedTaskResult(failed_result_content='I will implement the dictionary dispatch by creating a dictionary that maps operation strings to corresponding lambda functions. Here\'s the updated code:\n\n```python\ndef calculate(x, y, operation):\n    operations = {\n        "add": lambda a, b: a + b,\n        "subtract": lambda a, b: a - b,\n        "multiply": lambda a, b: a * b,\n        "divide": lambda a, b: a / b\n    }\n    return operations[operation](x, y)\n```\n\nIn this implementation, I have created a dictionary called `operations` that maps each operation string to a lambda function that performs the corresponding operation. Then, I use the dictionary to look up the appropriate function based on the `operation` parameter and call it with the `x` and `y` parameters. This replaces the original if-elif chain with a more concise and scalable approach using dictionary dispatch.', feedback='Good start, but add error handling: raise ValueError for unknown operations and ZeroDivisionE

In [11]:
step = await task_handler.get_next_step(rejected)
step

INFO (llm_agents_fs.SupervisedTaskHandler) :      🧠 New Step (rejection): Good start, but add error handling: raise ValueError for unknown operations and ZeroDivisionError for division by zero.


TaskStep(id_='7f8ccce0-4a9b-4df9-b04c-cbbc85405f10', task_id='830884ad-95a4-4fd7-9002-67deb56e4fba', instruction='The human operator REJECTED your proposed task result. Revise your approach and try again.\n\n<proposed-result>\nI will implement the dictionary dispatch by creating a dictionary that maps operation strings to corresponding lambda functions. Here\'s the updated code:\n\n```python\ndef calculate(x, y, operation):\n    operations = {\n        "add": lambda a, b: a + b,\n        "subtract": lambda a, b: a - b,\n        "multiply": lambda a, b: a * b,\n        "divide": lambda a, b: a / b\n    }\n    return operations[operation](x, y)\n```\n\nIn this implementation, I have created a dictionary called `operations` that maps each operation string to a lambda function that performs the corresponding operation. Then, I use the dictionary to look up the appropriate function based on the `operation` parameter and call it with the `x` and `y` parameters. This replaces the original if-

In [12]:
step_result = await task_handler.run_step(step)
step_result

INFO (llm_agents_fs.SupervisedTaskHandler) :      ⚙️ Processing Step: The human operator REJECTED your proposed task result. Revise your approach and try again.

<proposed-result>
I will implement the...[TRUNCATED]
INFO (llm_agents_fs.SupervisedTaskHandler) :      ✅ Step Result: I need to revise my approach by adding error handling to the dictionary dispatch implementation. Specifically, I should raise a `ValueE...[TRUNCATED]


TaskStepResult(task_step_id='7f8ccce0-4a9b-4df9-b04c-cbbc85405f10', content='I need to revise my approach by adding error handling to the dictionary dispatch implementation. Specifically, I should raise a `ValueError` if the provided operation is not recognized and a `ZeroDivisionError` if the division operation is used and the second operand (`y`) is zero. Let me update the code accordingly.\n\nHere\'s the revised implementation:\n\n```python\ndef calculate(x, y, operation):\n    operations = {\n        "add": lambda a, b: a + b,\n        "subtract": lambda a, b: a - b,\n        "multiply": lambda a, b: a * b,\n        "divide": lambda a, b: a / b if b != 0 else (raise ZeroDivisionError("Division by zero"))\n    }\n    \n    if operation not in operations:\n        raise ValueError(f"Unknown operation: {operation}")\n    \n    try:\n        return operations[operation](x, y)\n    except ZeroDivisionError as e:\n        raise e\n```\n\nIn this updated version, I have added a check to e

In [13]:
step = await task_handler.get_next_step(step_result)
step

In [14]:
step_result = await task_handler.run_step(step)
step_result

In [15]:
result = await task_handler.get_next_step(step_result)
result

In [16]:
await task_handler.complete(result)
print(result.content)

You're absolutely right. The `raise` statement cannot be used directly inside a lambda function in Python. To handle the division by zero case properly, I should use a separate function for the "divide" operation and add a check outside the lambda. Let me revise the code accordingly.

Here's the updated implementation:

```python
def calculate(x, y, operation):
    def divide(a, b):
        if b == 0:
            raise ZeroDivisionError("Division by zero")
        return a / b

    operations = {
        "add": lambda a, b: a + b,
        "subtract": lambda a, b: a - b,
        "multiply": lambda a, b: a * b,
        "divide": divide
    }

    if operation not in operations:
        raise ValueError(f"Unknown operation: {operation}")

    try:
        return operations[operation](x, y)
    except ZeroDivisionError as e:
        raise e
```

In this revised version:
1. I defined a separate function `divide` to handle the division operation and check for division by zero.
2. I updated t

## Trajectory 2: Abort

The agent is asked to refactor the authentication module. After seeing the
first planned step, the human realises the task conflicts with an ongoing
security audit and calls `abort()`. The handler terminates cleanly and the
caller can inspect the exception that was set.

In [17]:
task2 = Task(
    instruction=(
        "Refactor the authentication module to consolidate all login, "
        "logout, and session renewal functions into a single AuthService class."
    ),
)
task_handler2 = await agent.run_supervised(task2)

In [18]:
step = await task_handler2.get_next_step(None)
step

TaskStep(id_='6faf3aa0-4290-4e79-b860-47d2a9c76bbd', task_id='18f81a99-5f1f-42d9-babd-dd19bfad4953', instruction='Refactor the authentication module to consolidate all login, logout, and session renewal functions into a single AuthService class.')

In [19]:
await task_handler2.abort(
    error=RuntimeError(
        "Refactoring auth is blocked pending the security audit. "
        "Revisit after the audit completes.",
    ),
)

In [20]:
task_handler2.exception()

RuntimeError('Refactoring auth is blocked pending the security audit. Revisit after the audit completes.')